# Goal

Тестируем `18d_world_model_05`, а именно:
1) факторизованную `WorldModel`: `ReconstructionModel` (spatial, structure) + `PredictionModel` (temporal, dynamics)
2) отчуждаемый `ReconstructionModel`
3) в `ReconstructionModel` используются раздельные `lv_encoding` (как это было в `18d_world_model_03`)
4) `model.render_head.render_engine(type='layer', layers_count=4, features_counts=...)`
5) `with_prediction=False`
6) БЕЗ `BatchNorm2d` в рендерере
7) **dataset БЕЗ макспулинга** (чтобы не было соплей на картинках)

# TARGET_NOTEBOOK_FNAME

In [26]:
TARGET_NOTEBOOK_FNAME = '18d_world_model_05.ipynb'

# GRID_SEARCH_SPACE

In [27]:
# @launchit.collect
# GRID_SEARCH_SPACE = dict(
# )

# set_hyperparameters

In [29]:
# @launchit.collect
def set_hyperparameters(HP, optuna_study, optuna_trial):
    import random
    HP.general.comment = None
    HP.general.random_seed = random.randint(0, 100)
    HP.general.is_torch_deterministic = True
    HP.general.is_torch_compile = True
    HP.general.is_torch_amp = True
    
    HP.dataset.train = [
        'train_dataset_seq_eq_5_max_eq_0:101',
        'train_dataset_seq_eq_5_max_eq_0:102',
        'train_dataset_seq_eq_5_max_eq_0:103',
        'train_dataset_seq_eq_5_max_eq_0:104',
        'train_dataset_seq_eq_5_max_eq_0:105',
        'train_dataset_seq_eq_5_max_eq_0:106',
        'train_dataset_seq_eq_5_max_eq_0:107',
        'train_dataset_seq_eq_5_max_eq_0:108',
        'train_dataset_seq_eq_5_max_eq_0:109',
        'train_dataset_seq_eq_5_max_eq_0:110',
    ]
    HP.dataset.test = 'test_dataset_seq_eq_5_max_eq_0:101'

    HP.model.parent = None
    HP.model.sequence_length = 4
    HP.model.actions_count = 6
    HP.model.ob_shape = (1, 178, 152)
    HP.model.d_model = 256
    HP.model.vision_head = dict(grid=(6,6), features_counts=(16, 32, 64, 128))
    HP.model.transformer = dict(layers_count=3, heads_count=4, attention_backend=['EFFICIENT_ATTENTION', 'MATH'])

    features_counts = [
        (64, 48, 32, 16), 
        (96, 64, 32, 16), 
    ][random.randint(0, 1)]
    
    HP.model.render_head = dict(
        projector=dict(type='linear'), 
        render_engine=dict(
            type='layer', 
            layers_count=4, 
            features_counts=features_counts,
            is_batch_norm=False,
        )
    )
    
    HP.train.epochs_count = 100
    HP.train.batch_size = 128
    HP.train.optimizer = 'AdamW'
    HP.train.max_grad_norm = 1.0
    HP.train.learn_rate = 'const(0.00025)'
    HP.train.bce_loss_coef = 'const(1.0)'
    HP.train.edge_loss_coef = 'const(1.0)'
    HP.train.recon_loss_coef = 'const(1.0)'
    HP.train.pred_loss_coef = 'const(0)'
    HP.train.reg_loss_coefs = {}
    HP.train.with_prediction = False
    
    return HP

# Results

Лучший `ssim=0.9827` (`18d_world_model_05/opt_13.1/258`) с `features_counts=(96, 64, 32, 16)` (ожидаемо). Это ниже, чем `ssim=0.9893`, который был у `18d_world_model_05/opt_10.1/166`. 

Гипотеза: макс пулинг частенько размывает картинку, что делает задачу восстановления проще, т.к. становиться меньше четких граней. 

<img src="./img/ssim.png">

**Выводы**
1) думаю, что лучше без макс пулинга двигаться - так честнее
2) надо переходить к предсказанию

# System

In [30]:
import os, sys, re, subprocess, json
import IPython 
import concurrent.futures as cf
from collections import namedtuple

import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
from optuna.trial import TrialState

project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]

sys.path.append(os.path.join(project_root_path, 'lib'))

from logging_utils import *
from math_utils import *
from artifact_registry import *
import launchit
import launch_dispatcher
from autoincrement import Autoincrement

In [31]:
CONFIG = namedtuple('CONFIG', 
                    'project_root_uri, model_group_uri, project_root_path, subproject_name, subproject_path, run_path, ' + 
                    'target_notebook_fname, target_notebook_name, ' + 
                    'optuna_study_notebook_fname, optuna_study_name, optuna_study_serial, optuna_study_fname')(
    project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
    model_group_uri=None,
    project_root_path=project_root_path,
    subproject_name=None,
    subproject_path=os.path.abspath('../..'),
    run_path=None,
    target_notebook_fname=os.path.join(os.path.abspath('../..'), TARGET_NOTEBOOK_FNAME),
    target_notebook_name=None,
    optuna_study_notebook_fname=None,
    optuna_study_name=None,
    optuna_study_serial=None,
    optuna_study_fname=None,
)

with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as connection_file:
    optuna_study_notebook_fname = json.load(connection_file).get('jupyter_session')
    optuna_study_name, _ = os.path.splitext(os.path.basename(optuna_study_notebook_fname))
    optuna_study_serial = re.match(r'\w+_([\d\.\w]+)', optuna_study_name).group(1)
    optuna_study_fname = os.path.join(os.path.dirname(optuna_study_notebook_fname), optuna_study_name + '.optuna')
    CONFIG = CONFIG._replace(optuna_study_notebook_fname=optuna_study_notebook_fname)
    CONFIG = CONFIG._replace(optuna_study_name=optuna_study_name)
    CONFIG = CONFIG._replace(optuna_study_serial=optuna_study_serial)
    CONFIG = CONFIG._replace(optuna_study_fname=optuna_study_fname)

target_notebook_name, _ = os.path.splitext(os.path.basename(TARGET_NOTEBOOK_FNAME))
CONFIG = CONFIG._replace(subproject_name=os.path.basename(os.path.dirname(CONFIG.target_notebook_fname)))
CONFIG = CONFIG._replace(model_group_uri=f'{CONFIG.project_root_uri}.{CONFIG.subproject_name}')
CONFIG = CONFIG._replace(target_notebook_name=target_notebook_name)
CONFIG = CONFIG._replace(run_path=os.path.join(project_root_path, 'run', CONFIG.subproject_name))
CONFIG._asdict()

{'project_root_uri': 'com.develorium.neurolab',
 'model_group_uri': 'com.develorium.neurolab.18_rl',
 'project_root_path': '/home/misha/dev/mine/neurolab',
 'subproject_name': '18_rl',
 'subproject_path': '/home/misha/dev/mine/neurolab/18_rl',
 'run_path': '/home/misha/dev/mine/neurolab/run/18_rl',
 'target_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/18d_world_model_05.ipynb',
 'target_notebook_name': '18d_world_model_05',
 'optuna_study_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18d_study_13.1/18d_study_13.1.ipynb',
 'optuna_study_name': '18d_study_13.1',
 'optuna_study_serial': '13.1',
 'optuna_study_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18d_study_13.1/18d_study_13.1.optuna'}

In [32]:
LOG = Logging.get()
LOG.enable('syslog', False)
LOG.enable('stdout', False)
LOG.enable('verbose_stdout', True)

In [33]:
ARTIFACT_REGISTRY = ArtifactRegistry(maven_group_id=CONFIG.model_group_uri)

In [34]:
def create_optuna_launch():
    model_version = int(Autoincrement.get(f'{CONFIG.model_group_uri}.{CONFIG.target_notebook_name}'))
    assert model_version > 0, model_version
    ARTIFACT_REGISTRY.register_component(CONFIG.target_notebook_name, model_version)
    LOG(f'Model instance registered, version={model_version}')
    
    # Prep docker launch
    expandvars = dict(
        PROJECT_ROOT_PATH='/neurolab',
        BUILD_PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_GROUP_URI=CONFIG.model_group_uri,
        MODEL_NAME=CONFIG.target_notebook_name,
        MODEL_VERSION=model_version,
        LAUNCH_GOAL='TRAIN',
        OPTUNA_STUDY_FNAME=CONFIG.optuna_study_fname,
        OPTUNA_STUDY_NAME=CONFIG.optuna_study_name,
    )
    launch_fname = launchit.launchit(
        CONFIG.target_notebook_fname, 
        launch_serial=int(model_version),
        expandvars=expandvars, 
        make_py_file=False, 
        dir_name=CONFIG.run_path,
        collect_inds=['temp_config', 'optuna', 'initrd', 'build_docker_launch', 'optuna_run_docker_launch'],
        disable_inds=[],
    )
    return f'{CONFIG.target_notebook_name}:{model_version}', launch_fname

In [35]:
# Executed in a separate thread with GIL locked
def run_optuna_launch(launch_fname):
    # Run launch notebook locally, the latter will:
    # 1) sample values of hyperparameters from optuna study
    # 2) pack everything to docker launch (self-contained thing)
    # 3) run "docker_launch_run" cell which in turn will dispatch launch to cloud via launch_dispatcher
    # 4) collect result of a docker launch from cloud and update optuna study
    LOG(f'Launching "{launch_fname}"')
    
    subprocess.run(
        ['papermill', launch_fname, launch_fname, '--no-progress-bar'],
        capture_output=False,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True,
    )

    if os.path.exists(launch_fname + '.out'):
        with open(launch_fname + '.out', 'rt') as f:
            LOG(f.read())
    else:
        LOG(f'"{launch_fname}" completed with no output, probably failed')

## Unleash!

In [36]:
optuna_study = optuna.create_study(
    study_name=CONFIG.optuna_study_name,
    directions=['maximize'],
    storage=JournalStorage(JournalFileBackend(file_path=CONFIG.optuna_study_fname)),
    load_if_exists=True,
)
optuna_study.set_user_attr('STUDY_SERIAL', CONFIG.optuna_study_serial)
launches_count = 40
completed_launches_count = 0

with LOG.auto_log_level(logging.INFO):
    with cf.ThreadPoolExecutor(max_workers=32) as executor:
        futures = {}
        idle_runners_af = RecursiveMovingAverageFilter(max_n=6)
        is_first_time = True
        
        while launches_count is None or completed_launches_count < launches_count:
            runners_info = launch_dispatcher.RunnersInfo.get()
            idle_runners_af(runners_info['idle'])

            if is_first_time or (idle_runners_af.n >= idle_runners_af.max_n and idle_runners_af.v >= 1):
                if launches_count is None or (completed_launches_count + len(futures) < launches_count):
                    launch_name, launch_fname = create_optuna_launch()
                    futures.update({executor.submit(run_optuna_launch, launch_fname): launch_name})
                    LOG(f'{idle_runners_af.v:.1f} idle runners exist, submitted launch "{launch_name}"; running launches={len(futures)}')
                    idle_runners_af.reset()
                    
                is_first_time = False

            try:
                while futures:
                    completed_futures, _ = cf.wait(futures, timeout=0.1, return_when=cf.FIRST_COMPLETED)

                    if not completed_futures:
                        break
                        
                    for completed_future in completed_futures:
                        launch_name = futures[completed_future]
                        del futures[completed_future]

                        exc = completed_future.exception()
                        
                        if exc is not None:
                            LOG(f'Launch "{launch_name}" failed: {exc}')
                        else:
                            LOG(f'Launch "{launch_name}" completed')
    
                    if completed_futures:
                        completed_launches_count += len(completed_futures)
                        LOG(f'{completed_launches_count} (+{len(completed_futures)}) launches completed; running launches={len(futures)}')
            except TimeoutError as e:
                pass

            time.sleep(5)

[I 2026-09-07 21:50:52,009] A new study created in Journal with name: 18d_study_13.1


2026.09.07-21:50:52.915572     0.177 >> Model instance registered, version=239
2026.09.07-21:50:52.940898     0.009 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18d_world_model_05-launch239.ipynb"
2026.09.07-21:50:52.941258     0.010 >> 2.0 idle runners exist, submitted launch "18d_world_model_05:239"; running launches=1
2026.09.07-21:51:23.956312     0.194 >> Model instance registered, version=240
2026.09.07-21:51:23.973188     0.009 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18d_world_model_05-launch240.ipynb"
2026.09.07-21:51:23.973996     0.010 >> 1.3 idle runners exist, submitted launch "18d_world_model_05:240"; running launches=2
2026.09.07-22:07:56.079977     0.144 >> Model instance registered, version=241
2026.09.07-22:07:56.095922     0.007 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18d_world_model_05-launch241.ipynb"
2026.09.07-22:07:56.096168     0.008 >> 1.0 idle runners exist, submitted launch "18d_world_model_05:241"; running launches=3
2026

KeyboardInterrupt: 